In [1]:
import pandas as pd


def get_data(split):
    df = pd.read_csv(f"../hw1/wikIR1k/{split}/queries.csv")
    queries = dict(zip(df["id_left"], df["text_left"]))

    relevant_docs = {}
    with open(f"../hw1/wikIR1k/{split}/qrels", "r") as f:
        for line in f:
            query_id, _, doc_id, _ = line.strip().split()
            relevant_docs.setdefault(query_id, []).append(doc_id)
    documents = pd.read_csv("../hw1/wikIR1k/documents.csv")
    return documents, relevant_docs, queries


documents, relevant_docs, queries = get_data("test")

In [2]:
def basic_stats(queries, relevant_docs):
    result = {}
    result["num_queries"] = len(relevant_docs)

    q_lenghts = [len(query.split()) for query in queries.values()]
    result["max_query_length_in_words"] = max(q_lenghts)
    result["min_query_length_in_words"] = min(q_lenghts)
    result["avg_query_length_in_words"] = sum(q_lenghts) / len(q_lenghts)
    result["median_query_length_in_words"] = sorted(q_lenghts)[len(q_lenghts) // 2]

    docs_per_query = [len(docs) for docs in relevant_docs.values()]
    result["max_relevant_docs_per_query"] = max(docs_per_query)
    result["min_relevant_docs_per_query"] = min(docs_per_query)
    result["avg_relevant_docs_per_query"] = sum(docs_per_query) / len(docs_per_query)
    result["median_relevant_docs_per_query"] = sorted(docs_per_query)[len(docs_per_query) // 2]

    return result


basic_stats(queries, relevant_docs)

{'num_queries': 100,
 'max_query_length_in_words': 9,
 'min_query_length_in_words': 1,
 'avg_query_length_in_words': 2.07,
 'median_query_length_in_words': 2,
 'max_relevant_docs_per_query': 2759,
 'min_relevant_docs_per_query': 6,
 'avg_relevant_docs_per_query': 44.35,
 'median_relevant_docs_per_query': 9}

In [3]:
ind2id = dict(enumerate(documents["id_right"]))
id2ind = {v: k for k, v in ind2id.items()}
topk = 100

In [4]:
import time

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm


def process_bm25(queries, documents, ind2id, topk, tokenizer: str):
    prefill_start = time.time()
    bm25 = BM25Okapi(x.split() for x in documents.text_right)
    prefill_time = time.time() - prefill_start
    print(f"BM25 prefill time: {prefill_time:.2f} seconds")

    results = {}

    processing_start = time.time()
    for query_id, query in tqdm(queries.items()):
        tokenized_query = query.split(" ")
        doc_scores = bm25.get_scores(tokenized_query)

        top_doc_indices = doc_scores.argsort()[-topk:][::-1]
        top_doc_ids = [(ind2id[idx], doc_scores[idx]) for idx in top_doc_indices]

        results[query_id] = top_doc_ids

    processing_time = time.time() - processing_start
    print(f"BM25 avg processing time: {processing_time / len(queries):.2f} seconds")

    with open(f"bm25_{tokenizer}.res", "w") as f:
        for query_id, doc_scores in results.items():
            for rank, (doc_id, score) in enumerate(doc_scores):
                f.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} BM25\n")

In [5]:
def process_tfidf(queries, documents, ind2id, topk, tokenizer: str):
    prefill_start = time.time()
    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
    )
    doc_matrix = vectorizer.fit_transform(documents.text_right)
    prefill_time = time.time() - prefill_start
    print(f"TF-IDF prefill time: {prefill_time:.2f} seconds")

    results = {}

    processing_start = time.time()
    for query_id, query in tqdm(queries.items()):
        query_vec = vectorizer.transform([query])
        doc_scores = (doc_matrix @ query_vec.T).toarray().ravel()

        top_doc_indices = doc_scores.argsort()[-topk:][::-1]
        top_doc_ids = [(ind2id[idx], float(doc_scores[idx])) for idx in top_doc_indices]

        results[query_id] = top_doc_ids

    processing_time = time.time() - processing_start
    print(f"TF-IDF avg processing time: {processing_time / len(queries):.2f} seconds")

    with open(f"tfidf_{tokenizer}.res", "w") as f:
        for query_id, doc_scores in results.items():
            for rank, (doc_id, score) in enumerate(doc_scores):
                f.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} TFIDF\n")


In [8]:
documents, relevant_docs, queries = get_data("test")

process_bm25(queries, documents, ind2id, topk, tokenizer="original")
process_tfidf(queries, documents, ind2id, topk, tokenizer="original")

BM25 prefill time: 18.63 seconds


  0%|          | 0/100 [00:00<?, ?it/s]

BM25 avg processing time: 0.25 seconds
TF-IDF prefill time: 17.58 seconds


  0%|          | 0/100 [00:00<?, ?it/s]

TF-IDF avg processing time: 0.11 seconds


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()


def stem_text(text: str) -> str:
    return " ".join(stemmer.stem(word) for word in text.split())


documents, relevant_docs, queries = get_data("test")
documents["text_right"] = [stem_text(text) for text in tqdm(documents["text_right"], total=len(documents))]
queries = {qid: stem_text(query) for qid, query in queries.items()}

process_bm25(queries, documents, ind2id, topk, tokenizer="stemmed")
process_tfidf(queries, documents, ind2id, topk, tokenizer="stemmed")

  0%|          | 0/369721 [00:00<?, ?it/s]

BM25 prefill time: 14.38 seconds


  0%|          | 0/100 [00:00<?, ?it/s]

BM25 avg processing time: 0.27 seconds
TF-IDF prefill time: 15.95 seconds


  0%|          | 0/100 [00:00<?, ?it/s]

TF-IDF avg processing time: 0.13 seconds


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

documents, relevant_docs, queries = get_data("test")


def lemmatize_texts(texts):
    lemmatized = []
    for doc in tqdm(nlp.pipe(texts, batch_size=256, n_process=8), total=len(texts)):
        lemmatized.append(" ".join(token.lemma_ for token in doc))
    return lemmatized


documents["text_right"] = lemmatize_texts(documents["text_right"].tolist())
queries = dict(zip(queries.keys(), lemmatize_texts(list(queries.values()))))

process_bm25(queries, documents, ind2id, topk, tokenizer="lemmatized")
process_tfidf(queries, documents, ind2id, topk, tokenizer="lemmatized")

  0%|          | 0/100 [00:00<?, ?it/s]

BM25 prefill time: 19.87 seconds


  0%|          | 0/100 [00:00<?, ?it/s]

BM25 avg processing time: 0.24 seconds
TF-IDF prefill time: 15.29 seconds


  0%|          | 0/100 [00:00<?, ?it/s]

TF-IDF avg processing time: 0.12 seconds


In [6]:
import ir_measures
from IPython.display import display
from ir_measures import MAP, P, nDCG

qrels_path = "../hw1/wikIR1k/test/qrels"
run_paths = {
    "BM25_original": "bm25_original.res",
    "BM25_stemmed": "bm25_stemmed.res",
    "BM25_lemmatized": "bm25_lemmatized.res",
    "TF-IDF_original": "tfidf_original.res",
    "TF-IDF_stemmed": "tfidf_stemmed.res",
    "TF-IDF_lemmatized": "tfidf_lemmatized.res",
}

measure_specs = [
    ("P@1", P(cutoff=1, rel=1, judged_only=False), "P(cutoff=1, rel=1, judged_only=False)"),
    ("P@10", P(cutoff=10, rel=1, judged_only=False), "P(cutoff=10, rel=1, judged_only=False)"),
    ("P@20", P(cutoff=20, rel=1, judged_only=False), "P(cutoff=20, rel=1, judged_only=False)"),
    ("MAP", MAP(rel=1, judged_only=False), "MAP(cutoff=not set, rel=1, judged_only=False)"),
    (
        "nDCG@20",
        nDCG(cutoff=20, dcg="log2", judged_only=False),
        "nDCG(cutoff=20, dcg='log2', gains=not set, judged_only=False)",
    ),
]

print("Measures used:")
for name, _, params in measure_specs:
    print(f"- {name}: {params}")

rows = []
for run_name, run_path in run_paths.items():
    qrels = ir_measures.read_trec_qrels(qrels_path)
    run = ir_measures.read_trec_run(run_path)
    aggregate = ir_measures.calc_aggregate([measure for _, measure, _ in measure_specs], qrels, run)

    row = {"Run": run_name}
    for label, measure, _ in measure_specs:
        row[label] = aggregate[measure]
    rows.append(row)

results_df = pd.DataFrame(rows)

display(
    results_df.style.format(
        {
            "P@1": "{:.4f}",
            "P@10": "{:.4f}",
            "P@20": "{:.4f}",
            "MAP": "{:.4f}",
            "nDCG@20": "{:.4f}",
        }
    )
)

results_df

Measures used:
- P@1: P(cutoff=1, rel=1, judged_only=False)
- P@10: P(cutoff=10, rel=1, judged_only=False)
- P@20: P(cutoff=20, rel=1, judged_only=False)
- MAP: MAP(cutoff=not set, rel=1, judged_only=False)
- nDCG@20: nDCG(cutoff=20, dcg='log2', gains=not set, judged_only=False)


,Run,P@1,P@10,P@20,MAP,nDCG@20
0,BM25_original,0.4900,0.2120,0.1500,0.1704,0.3571
1,BM25_stemmed,0.4900,0.2090,0.1455,0.1689,0.3540
2,BM25_lemmatized,0.4800,0.2100,0.1480,0.1707,0.3561
3,TF-IDF_original,0.5100,0.2030,0.1350,0.1604,0.3498
4,TF-IDF_stemmed,0.5000,0.1940,0.1285,0.1542,0.3353
5,TF-IDF_lemmatized,0.4800,0.2000,0.1300,0.1566,0.3392


,Run,P@1,P@10,P@20,MAP,nDCG@20
0,BM25_original,0.49,0.212,0.1500,0.170363,0.357085
1,BM25_stemmed,0.49,0.209,0.1455,0.168926,0.354021
2,BM25_lemmatized,0.48,0.210,0.1480,0.170681,0.356117
3,TF-IDF_original,0.51,0.203,0.1350,0.160372,0.349805
4,TF-IDF_stemmed,0.50,0.194,0.1285,0.154246,0.335334
5,TF-IDF_lemmatized,0.48,0.200,0.1300,0.156642,0.339170


In [ ]:
from ir_measures import AP

documents, relevant_docs, queries = get_data("test")
documents["id_right"] = documents["id_right"].astype(str)
queries_df = pd.read_csv("../hw1/wikIR1k/test/queries.csv")
queries_df["id_left"] = queries_df["id_left"].astype(str)
doc_text = dict(zip(documents["id_right"], documents["text_right"]))
qrels = list(ir_measures.read_trec_qrels(qrels_path))

analysis_measures = {
    "P@1": P(cutoff=1, rel=1, judged_only=False),
    "P@10": P(cutoff=10, rel=1, judged_only=False),
    "P@20": P(cutoff=20, rel=1, judged_only=False),
    "AP": AP(rel=1, judged_only=False),
    "nDCG@20": nDCG(cutoff=20, dcg="log2", judged_only=False),
}


def per_query_metrics(run_path):
    rows = []
    run = list(ir_measures.read_trec_run(run_path))
    for metric in ir_measures.iter_calc(list(analysis_measures.values()), qrels, run):
        rows.append((metric.query_id, str(metric.measure), float(metric.value)))
    return (
        pd.DataFrame(rows, columns=["query_id", "measure", "value"])
        .pivot(index="query_id", columns="measure", values="value")
        .reset_index()
    )


bm25_query_df = per_query_metrics("bm25_original.res")
analysis_df = queries_df.rename(columns={"id_left": "query_id", "text_left": "query"}).merge(
    bm25_query_df, on="query_id"
)
analysis_df["query_len"] = analysis_df["query"].str.split().str.len()
analysis_df["rel_count"] = analysis_df["query_id"].map(lambda qid: len(relevant_docs[qid]))

easy_queries = analysis_df.sort_values("nDCG@20", ascending=False)[
    ["query_id", "query", "nDCG@20", "query_len", "rel_count"]
].head(5)
hard_queries = analysis_df.sort_values("nDCG@20", ascending=True)[
    ["query_id", "query", "nDCG@20", "query_len", "rel_count"]
].head(5)

print("BM25 original: easy queries")
display(easy_queries)
print("BM25 original: hard queries")
display(hard_queries)
print(f"Pearson corr(query length, nDCG@20) = {analysis_df['query_len'].corr(analysis_df['nDCG@20']):.4f}")
print(
    f"Spearman corr(# relevant docs, nDCG@20) = {analysis_df['rel_count'].corr(analysis_df['nDCG@20'], method='spearman'):.4f}"
)

length_summary = analysis_df.groupby("query_len")["nDCG@20"].agg(["count", "mean"]).reset_index()
length_summary["mean"] = length_summary["mean"].round(4)
length_summary = length_summary.rename(columns={"mean": "avg_nDCG@20"})
display(length_summary)

rel_summary = (
    analysis_df.assign(
        rel_bucket=pd.cut(
            analysis_df["rel_count"],
            bins=[0, 10, 50, 500, 3000],
            labels=["1-10", "11-50", "51-500", "501+"],
        )
    )
    .groupby("rel_bucket", observed=False)["nDCG@20"]
    .agg(["count", "mean"])
    .reset_index()
)
rel_summary["mean"] = rel_summary["mean"].round(4)
rel_summary = rel_summary.rename(columns={"mean": "avg_nDCG@20"})
display(rel_summary)

BM25 original: easy queries


,query_id,query,nDCG@20,query_len,rel_count
22,75295,dave matthews band,0.849280,3,8
29,267973,gambela region,0.822220,2,8
33,113000,sulfide,0.748263,1,6
70,14410,xml,0.730094,1,28
9,104086,bulacan,0.716428,1,10


BM25 original: hard queries


,query_id,query,nDCG@20,query_len,rel_count
14,83078,pomacentridae,0.000000,1,8
88,738573,television presenter,0.000000,2,59
72,1897191,centrism,0.000000,1,14
55,79032,elijah wood,0.000000,2,6
97,5622,homer,0.040379,1,19


Pearson corr(query length, nDCG@20) = 0.0503
Spearman corr(# relevant docs, nDCG@20) = -0.3208


,query_len,count,avg_nDCG@20
0,1,32,0.3588
1,2,46,0.3487
2,3,14,0.3767
3,4,5,0.3169
4,6,2,0.4392
5,9,1,0.4512


,rel_bucket,count,avg_nDCG@20
0,1-10,58,0.4070
1,11-50,35,0.2970
2,51-500,6,0.2467
3,501+,1,0.2303


In [7]:
def evaluate_bm25(queries, documents, qrels_path, ind2id, topk, k1, b):
    bm25 = BM25Okapi([x.split() for x in documents.text_right], k1=k1, b=b)
    results = {}

    for query_id, query in tqdm(queries.items()):
        tokenized_query = query.split(" ")
        doc_scores = bm25.get_scores(tokenized_query)

        top_doc_indices = doc_scores.argsort()[-topk:][::-1]
        top_doc_ids = [(ind2id[idx], doc_scores[idx]) for idx in top_doc_indices]

        results[query_id] = top_doc_ids

    run_path = "bm25_tmp.res"

    with open(run_path, "w") as f:
        for query_id, doc_scores in results.items():
            for rank, (doc_id, score) in enumerate(doc_scores):
                f.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} BM25\n")

    qrels = ir_measures.read_trec_qrels(qrels_path)
    run = ir_measures.read_trec_run(run_path)
    metric = nDCG(cutoff=20, dcg="log2", judged_only=False)
    return ir_measures.calc_aggregate([metric], qrels, run)[metric]

In [9]:
import math

documents, relevant_docs, queries = get_data("validation")
qrels_path = "../hw1/wikIR1k/validation/qrels"

best_score = (-math.inf, None, None)
for k1 in [0.5, 1.0, 1.5, 2.0]:
    for b in [0.0, 0.25, 0.5, 0.75, 1.0]:
        score = evaluate_bm25(queries, documents, qrels_path, ind2id, topk, k1, b)
        print(f"BM25 with k1={k1}, b={b} achieved nDCG@20: {score:.4f}")
        if score > best_score[0]:
            best_score = (score, k1, b)

print(f"Best BM25 parameters: k1={best_score[1]}, b={best_score[2]} with nDCG@20: {best_score[0]:.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

BM25 with k1=0.5, b=0.0 achieved nDCG@20: 0.3211


  0%|          | 0/100 [00:00<?, ?it/s]

BM25 with k1=0.5, b=0.25 achieved nDCG@20: 0.3184


  0%|          | 0/100 [00:00<?, ?it/s]

...

BM25 with k1=2.0, b=0.5 achieved nDCG@20: 0.3139


  0%|          | 0/100 [00:00<?, ?it/s]

BM25 with k1=2.0, b=0.75 achieved nDCG@20: 0.3139


  0%|          | 0/100 [00:00<?, ?it/s]

BM25 with k1=2.0, b=1.0 achieved nDCG@20: 0.3140
Best BM25 parameters: k1=0.5, b=0.0 with nDCG@20: 0.3211


In [10]:
documents, relevant_docs, queries = get_data("test")
qrels_path = "../hw1/wikIR1k/test/qrels"

score = evaluate_bm25(queries, documents, qrels_path, ind2id, topk, best_score[1], best_score[2])
print(f"BM25 with k1={best_score[1]}, b={best_score[2]} achieved nDCG@20: {score:.4f} on test set")

  0%|          | 0/100 [00:00<?, ?it/s]

BM25 with k1=0.5, b=0.0 achieved nDCG@20: 0.3493 on test set
